# Task A / Candidate B：数值数组指针单步探针

本 Notebook 只验证候选 B 是否适合成为正式深度任务，不生成正式数据集。

## 冻结的候选规格
- 索引范围：0–39，共 40 个位置。
- 每个位置存储一个有符号偏移量 \(\Delta[i]\)。
- 每一步执行：\(i_{t+1}=i_t+\Delta[i_t]\)。
- 先构造长度为 \(d+1\) 的无重复路径，再填充未访问位置。
- 本版偏移量绝对值固定为 10–39，确保使用两位数偏移；不取模、不裁剪，所有目标索引均在 0–39。
- 本轮只测 \(d=1\)：100 道配对题，DA 与 CoT 各 100 次，共 200 次请求。

## 为什么 DA 和 CoT 都测
上一轮双寄存器 Pilot 显示两种模式的单步正确率不能互相代替。因此本轮分别测量 \(p_{1,DA}\) 和 \(p_{1,CoT}\)，不从其中一个推算另一个。

**这仍然只是候选筛选，不是正式数据集，也不会自动进入深度实验。**

## 1. 生成器与参考模拟器

In [ ]:
import hashlib
import json
import math
import os
import random
import re
import subprocess
import sys
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

N = 40
MIN_ABS_DELTA = 10
DATA_SEED = 2026110801
MODEL_SEED = 2026110802

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
MODEL_KW = dict(
    dtype="half",
    max_model_len=8192,
    gpu_memory_utilization=0.90,
    max_num_seqs=32,
)
SAMPLING_BY_MODE = {
    "DA": dict(temperature=0.6, top_p=0.95, max_tokens=512),
    "CoT": dict(temperature=0.6, top_p=0.95, max_tokens=2048),
}
SYSTEM_PROMPTS = {
    "DA": (
        "Compute silently. Output exactly one line in this format: "
        "Final answer: <integer>. Do not explain or include any other text."
    ),
    "CoT": (
        "Work through the index transitions one at a time. For each transition, "
        "show the current index, the signed offset read from the table, and the "
        "resulting index. End with exactly one final line in this format: "
        "Final answer: <integer>. Do not repeat the final answer or write anything "
        "after that line."
    ),
}
SCORING_VERSION = "array-pointer-scalar-v1"
PROBE_VERSION = "task-a-candidate-b-p1-probe-v1"


def sha(value):
    payload = json.dumps(
        value, sort_keys=True, ensure_ascii=False, separators=(",", ":")
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def timestamp():
    return datetime.now(timezone.utc).isoformat()


def make_path(rng, depth, n=N, min_abs_delta=MIN_ABS_DELTA, max_restarts=10000):
    if type(depth) is not int or not 1 <= depth < n:
        raise ValueError("depth must be an integer in 1..N-1")
    if not 1 <= min_abs_delta < n:
        raise ValueError("min_abs_delta must be in 1..N-1")

    for _ in range(max_restarts):
        path = [rng.randrange(n)]
        visited = {path[0]}
        for _step in range(depth):
            current = path[-1]
            candidates = [
                j for j in range(n)
                if j not in visited and min_abs_delta <= abs(j - current) < n
            ]
            if not candidates:
                break
            nxt = rng.choice(candidates)
            path.append(nxt)
            visited.add(nxt)
        if len(path) == depth + 1:
            return path

    raise RuntimeError("Could not construct a non-repeating path")


def make_delta_array(rng, depth, n=N, min_abs_delta=MIN_ABS_DELTA):
    path = make_path(rng, depth, n, min_abs_delta)
    delta = [None] * n

    # 先按目标路径写入偏移量，保证指定起点会走过完整的 d 步。
    for current, nxt in zip(path, path[1:]):
        delta[current] = nxt - current

    # 再填充路径以外的位置；所有偏移量都指向合法索引。
    for i in range(n):
        if delta[i] is None:
            targets = [
                j for j in range(n)
                if j != i and min_abs_delta <= abs(j - i) < n
            ]
            delta[i] = rng.choice(targets) - i

    assert len(delta) == n
    assert all(type(x) is int for x in delta)
    assert all(0 <= i + delta[i] < n for i in range(n))
    assert all(min_abs_delta <= abs(x) < n for x in delta)
    return delta, path


def simulate(start, delta, depth):
    if type(start) is not int or not 0 <= start < len(delta):
        raise ValueError("start must be a valid integer index")
    current = start
    states = [current]
    for _ in range(depth):
        current = current + delta[current]
        if not 0 <= current < len(delta):
            raise AssertionError("transition left the valid index range")
        states.append(current)
    return states


def render_question(delta, start, depth):
    table = "\n".join(f"{i:02d}: {delta[i]:+d}" for i in range(len(delta)))
    return (
        "The table defines a signed-offset array indexed from 0 to 39. "
        "At each transition, read delta[i] from the row for the CURRENT index i, "
        "then update the index to i + delta[i]. Use the updated index for the next "
        "transition. Every resulting index is guaranteed to remain in 0..39; "
        "do not use modulo or clipping.\n\n"
        f"{table}\n\n"
        f"Start at index {start}. Apply exactly {depth} transitions.\n"
        "What is the final index?"
    )


def make_task(seed, depth, question_id=None):
    rng = random.Random(seed)
    delta, path = make_delta_array(rng, depth)
    states = simulate(path[0], delta, depth)
    if states != path:
        raise AssertionError("reference simulator disagrees with generated path")

    return {
        "question_id": question_id or f"array-b-d{depth}-s{seed}",
        "spec_version": "numeric-array-pointer-v1",
        "purpose": "candidate_b_p1_probe",
        "depth": depth,
        "start": path[0],
        "delta": delta,
        "path": path,
        "ans": states[-1],
        "W_transition": depth,
        "W_lookup": depth,
        "W_add": depth,
        "W": 2 * depth,
        "q": render_question(delta, path[0], depth),
    }


def make_probe_questions(n_questions=100):
    data_rng = random.Random(DATA_SEED)
    model_rng = random.Random(MODEL_SEED)
    questions = []

    for i in range(n_questions):
        question_seed = data_rng.getrandbits(63)
        model_seed = model_rng.randrange(2**31)
        task = make_task(
            question_seed,
            depth=1,
            question_id=f"array-b-v1-d1-i{i:03d}",
        )
        questions.append({
            **task,
            "question_seed": question_seed,
            "model_seed": model_seed,
            "question": task["q"],
        })

    signatures = [sha([q["start"], q["delta"]]) for q in questions]
    if len(set(signatures)) != n_questions:
        raise RuntimeError("Duplicate probe questions detected; do not resample silently")
    return questions

## 2. 最终答案解析与评分

只读取明确的 `Final answer:` 声明，不从中间步骤猜最终答案。相同的重复声明可以接受；冲突声明或畸形声明判为解析失败。

In [ ]:
_FINAL_MARKER = re.compile(r"^Final answer\s*[:：]\s*(.*)$", re.I)
_INTEGER = re.compile(r"^[+-]?\d+$")


def normalize_line(line):
    line = line.strip()
    line = re.sub(r"^#{1,6}\s*", "", line)
    line = re.sub(r"^\\*\\*(.*)\\*\\*$", r"\\1", line)
    line = re.sub(r"\\*\\*([^*]+)\\*\\*", r"\\1", line)
    line = line.replace("`", "").strip()
    return line


def parse_final_answer(raw_response, finish_reason=None):
    if not isinstance(raw_response, str):
        raise TypeError("raw_response must be a string")
    if finish_reason not in (None, "stop", "length", "abort", "error"):
        raise ValueError("unsupported finish_reason")

    if finish_reason == "error":
        return dict(
            answer_parse_ok=False, parsed_answer=None,
            parse_reason="runtime_error", ambiguity=False,
            final_declaration_count=0,
        )

    lines = [
        (i, normalize_line(line))
        for i, line in enumerate(raw_response.splitlines())
        if line.strip()
    ]
    declarations = []
    malformed = False

    for pos, (line_index, line) in enumerate(lines):
        match = _FINAL_MARKER.fullmatch(line)
        if not match:
            continue

        body = normalize_line(match.group(1))
        if not body and pos + 1 < len(lines):
            next_line = lines[pos + 1][1]
            if _INTEGER.fullmatch(next_line):
                body = next_line

        # 允许嵌套重复的 Final answer 标记，但最终内容必须是一个整数。
        body = re.sub(r"^Final answer\s*[:：]\s*", "", body, flags=re.I)
        body = normalize_line(body)
        if not _INTEGER.fullmatch(body):
            malformed = True
            continue
        declarations.append((line_index, int(body)))

    if not declarations:
        reason = "malformed_final_declaration" if malformed else "no_final_declaration"
        return dict(
            answer_parse_ok=False, parsed_answer=None,
            parse_reason=reason, ambiguity=False,
            final_declaration_count=0,
        )

    values = {value for _, value in declarations}
    if len(values) > 1:
        return dict(
            answer_parse_ok=False, parsed_answer=None,
            parse_reason="conflicting_final_declarations", ambiguity=True,
            final_declaration_count=len(declarations),
        )

    if malformed:
        return dict(
            answer_parse_ok=False, parsed_answer=None,
            parse_reason="malformed_final_declaration", ambiguity=False,
            final_declaration_count=len(declarations),
        )

    return dict(
        answer_parse_ok=True,
        parsed_answer=declarations[-1][1],
        parse_reason=(
            "identical_final_declarations" if len(declarations) > 1
            else "explicit_final"
        ),
        ambiguity=False,
        final_declaration_count=len(declarations),
    )


def score_response(raw_response, gold, finish_reason):
    parsed = parse_final_answer(raw_response, finish_reason)
    parsed_ok = parsed["answer_parse_ok"]
    answer_ok = bool(parsed_ok and parsed["parsed_answer"] == gold)
    return {
        **parsed,
        "scoring_version": SCORING_VERSION,
        "answer_ok": answer_ok,
        "parse_failure": not parsed_ok,
        "truncation": finish_reason == "length",
        "runtime_failure": finish_reason in ("abort", "error"),
    }


def wilson95(correct, total):
    if total == 0:
        return None
    z = 1.959963984540054
    rate = correct / total
    denominator = 1 + z*z/total
    centre = (rate + z*z/(2*total)) / denominator
    half = z * math.sqrt(
        rate*(1-rate)/total + z*z/(4*total*total)
    ) / denominator
    return [max(0.0, centre-half), min(1.0, centre+half)]

## 3. CPU 测试

先验证生成器、参考答案、无重复路径、合法索引和评分边界。此单元不调用模型。

In [ ]:
import unittest


class CandidateBTests(unittest.TestCase):
    def test_reproducibility(self):
        a = make_task(12345, 20)
        b = make_task(12345, 20)
        self.assertEqual(a, b)

    def test_depths_and_path_invariants(self):
        for depth in (1, 5, 10, 15, 20):
            task = make_task(2026110800 + depth, depth)
            self.assertEqual(len(task["path"]), depth + 1)
            self.assertEqual(len(set(task["path"])), depth + 1)
            self.assertEqual(simulate(task["start"], task["delta"], depth), task["path"])
            self.assertEqual(task["ans"], task["path"][-1])
            self.assertEqual(task["W"], 2 * depth)
            self.assertTrue(all(0 <= i + task["delta"][i] < N for i in range(N)))
            self.assertTrue(all(MIN_ABS_DELTA <= abs(x) < N for x in task["delta"]))

    def test_probe_count_and_unique_questions(self):
        questions = make_probe_questions(100)
        self.assertEqual(len(questions), 100)
        self.assertEqual(len({q["question_id"] for q in questions}), 100)
        self.assertTrue(all(q["depth"] == 1 for q in questions))

    def test_final_answer_parser(self):
        cases = [
            ("Final answer: 17", 17, True),
            ("**Final answer: 17**", 17, True),
            ("Final answer:\n**17**", 17, True),
            ("Step 1: 12 + 5 = 17\nFinal answer: 17", 17, True),
            ("Final answer: 17\nFinal answer: 17", 17, True),
            ("Final answer: 17\nFinal answer: 18", 17, False),
            ("Step 1: 17", 17, False),
            ("Final answer:", 17, False),
        ]
        for raw, gold, expected in cases:
            with self.subTest(raw=raw):
                self.assertEqual(score_response(raw, gold, "stop")["answer_ok"], expected)

    def test_conflicting_declarations_are_ambiguous(self):
        result = parse_final_answer("Final answer: 17\nFinal answer: 18", "stop")
        self.assertTrue(result["ambiguity"])
        self.assertFalse(result["answer_parse_ok"])


unittest.main(argv=[""], exit=False)

## 4. 请求记录、断点保护与配对消息

每道题分别发送 DA 和 CoT。原始响应先落盘，再评分。已发送但结果未知的请求不自动重发。

In [ ]:
def read_rows(path):
    path = Path(path)
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]


def append_row(path, row):
    with Path(path).open("a") as stream:
        stream.write(json.dumps(row, ensure_ascii=False) + "\n")
        stream.flush()
        os.fsync(stream.fileno())


def save_once(path, content):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        if path.read_text() != content:
            raise RuntimeError(f"Existing file differs; refusing to overwrite: {path}")
    else:
        with path.open("x") as stream:
            stream.write(content)
            stream.flush()
            os.fsync(stream.fileno())


def messages_for(question, mode):
    return [
        {"role": "system", "content": SYSTEM_PROMPTS[mode]},
        {"role": "user", "content": question["question"]},
    ]


def request_plan(questions):
    return [
        (question, mode)
        for question in questions
        for mode in ("DA", "CoT")
    ]


def request_status(out_dir):
    attempts = read_rows(Path(out_dir) / "request_attempts_v1.jsonl")
    results = read_rows(Path(out_dir) / "probe_results_v1.jsonl")
    attempt_ids = [r["request_id"] for r in attempts]
    result_ids = [r["request_id"] for r in results]

    if len(attempt_ids) != len(set(attempt_ids)):
        raise RuntimeError("Duplicate request attempts detected")
    if len(result_ids) != len(set(result_ids)):
        raise RuntimeError("Duplicate result records detected")
    if not set(result_ids).issubset(set(attempt_ids)):
        raise RuntimeError("A result has no matching request attempt")
    return attempts, results


def run_requests(out_dir, questions, manifest, runtime, generate_one, count_prompt, limit=None):
    out_dir = Path(out_dir)
    if runtime["protocol_sha256"] != manifest["protocol_sha256"]:
        raise RuntimeError("Runtime and probe protocol hashes differ")

    attempts, results = request_status(out_dir)
    attempted_ids = {r["request_id"] for r in attempts}
    result_ids = {r["request_id"] for r in results}
    if attempted_ids != result_ids:
        raise RuntimeError(
            "There are sent requests with unknown outcomes; do not retry or continue automatically"
        )

    if any(r["runtime_failure"] or r["truncation"] for r in results):
        raise RuntimeError("A runtime failure or truncation exists; stop for manual review")
    if (out_dir / "HALTED.json").exists():
        raise RuntimeError("HALTED.json exists; inspect it before continuing")

    planned = request_plan(questions)
    pending = [
        (q, mode) for q, mode in planned
        if f"{q['question_id']}-{mode}" not in result_ids
    ]
    if limit is not None:
        pending = pending[:limit]
    if not pending:
        print("No pending requests; no duplicate inference will be sent.")
        return results

    lock = out_dir / "run.lock"
    with lock.open("x") as stream:
        stream.write(timestamp())

    try:
        if request_status(out_dir) != (attempts, results):
            raise RuntimeError("Request state changed after acquiring lock")

        for question, mode in pending:
            request_id = f"{question['question_id']}-{mode}"
            messages = messages_for(question, mode)
            sampling = SAMPLING_BY_MODE[mode]
            prompt_tokens = count_prompt(messages)

            if prompt_tokens + sampling["max_tokens"] > MODEL_KW["max_model_len"]:
                raise RuntimeError("Prompt exceeds model context budget; request not sent")

            attempt = {
                "request_id": request_id,
                "question_id": question["question_id"],
                "depth": 1,
                "mode": mode,
                "seed": question["model_seed"],
                "question_seed": question["question_seed"],
                "start": question["start"],
                "delta": question["delta"],
                "gold_answer": question["ans"],
                "question": question["question"],
                "messages": messages,
                "protocol_sha256": manifest["protocol_sha256"],
                "model_id": MODEL_ID,
                "model_revision": runtime["model_revision"],
                "environment_sha256": runtime["environment_sha256"],
                "dispatched_at": timestamp(),
                "prompt_tokens": prompt_tokens,
            }
            append_row(out_dir / "request_attempts_v1.jsonl", attempt)

            raw_response = ""
            finish_reason = "error"
            completion_tokens = None
            generation_error = None
            started = time.perf_counter()
            try:
                generated = generate_one(messages, mode, question["model_seed"])
                raw_response = generated["raw_response"]
                finish_reason = generated["finish_reason"]
                prompt_tokens = generated["prompt_tokens"]
                completion_tokens = generated["completion_tokens"]
                if not isinstance(raw_response, str):
                    raise ValueError("raw_response must be a string")
                if finish_reason not in ("stop", "length", "abort", "error"):
                    raise ValueError("Unsupported finish_reason")
            except BaseException as exc:
                generation_error = f"{type(exc).__name__}: {exc}"
                finish_reason = "error"

            elapsed = time.perf_counter() - started
            generation = {
                **attempt,
                "prompt_tokens": prompt_tokens,
                "raw_response": raw_response,
                "finish_reason": finish_reason,
                "completion_tokens": completion_tokens,
                "elapsed_seconds": elapsed,
                "generation_error": generation_error,
                "received_at": timestamp(),
            }
            append_row(out_dir / "raw_generations_v1.jsonl", generation)

            score = score_response(raw_response, question["ans"], finish_reason)
            row = {
                **generation,
                **score,
                "raw_response_available": generation_error is None or bool(raw_response),
            }
            append_row(out_dir / "probe_results_v1.jsonl", row)
            results.append(row)

            if generation_error or finish_reason != "stop":
                reason = generation_error or f"finish_reason={finish_reason}"
                save_once(
                    out_dir / "HALTED.json",
                    json.dumps(
                        {"reason": reason, "request_id": request_id},
                        ensure_ascii=False, indent=2,
                    ) + "\n",
                )
                raise RuntimeError(f"Saved and halted: {reason}")

            print(
                f"{len(results)}/200 {request_id}: "
                f"answer={row['answer_ok']}, parse={row['answer_parse_ok']}, "
                f"tokens={completion_tokens}",
                flush=True,
            )

        return results
    finally:
        if lock.exists():
            lock.unlink()


def summarize_probe(records, planned_per_mode=100):
    summary = {}
    for mode in ("DA", "CoT"):
        rows = [r for r in records if r["mode"] == mode]
        attempted = len(rows)
        parsed = sum(r["answer_parse_ok"] for r in rows)
        correct = sum(r["answer_ok"] for r in rows)
        rate = correct / attempted if attempted else None
        summary[mode] = {
            "planned": planned_per_mode,
            "attempted": attempted,
            "not_attempted": planned_per_mode - attempted,
            "complete": attempted == planned_per_mode,
            "parsed": parsed,
            "parse_success_rate": parsed / attempted if attempted else None,
            "correct": correct,
            "main_accuracy": rate,
            "wilson95": wilson95(correct, attempted),
            "conditional_accuracy": correct / parsed if parsed else None,
            "parse_reasons": dict(Counter(r["parse_reason"] for r in rows)),
            "runtime_failures": sum(r["runtime_failure"] for r in rows),
            "truncations": sum(r["truncation"] for r in rows),
        }

    p1_da = summary["DA"]["main_accuracy"] if summary["DA"]["complete"] else None
    p1_cot = summary["CoT"]["main_accuracy"] if summary["CoT"]["complete"] else None
    summary["p1_DA"] = p1_da
    summary["p1_CoT"] = p1_cot
    summary["projected_CoT_d20"] = p1_cot ** 20 if p1_cot is not None else None
    summary["DA_d1_target"] = (
        0.80 <= p1_da <= 0.95 if p1_da is not None else None
    )
    summary["CoT_p1_reference_target"] = (
        0.955 <= p1_cot <= 0.986 if p1_cot is not None else None
    )
    summary["warning"] = (
        "The d=20 value is a model-based projection only, not an observed depth result. "
        "Do not generate the formal dataset until the probe and depth validation are reviewed."
    )
    return summary

## 5. CPU 回归测试

In [ ]:
suite = unittest.defaultTestLoader.loadTestsFromTestCase(CandidateBTests)
result = unittest.TextTestRunner(verbosity=2).run(suite)
if not result.wasSuccessful():
    raise RuntimeError("CPU regression tests failed")
print("CPU tests passed; model_requests=0")

## 6. Drive：冻结 100 道题与实验协议

此处只创建候选探针输入，不创建正式 d=1..20 数据集。若同名文件已存在且内容不同，会停止而不覆盖。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

OUT_DIR = Path(
    "/content/drive/MyDrive/CISC7404/Experiments/task_a/task_a_candidate_b_probe_v1"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

QUESTIONS = make_probe_questions(100)
dataset_text = "".join(
    json.dumps(q, ensure_ascii=False) + "\n" for q in QUESTIONS
)
save_once(OUT_DIR / "probe_dataset_v1.jsonl", dataset_text)

from datasets import load_dataset
loaded = load_dataset(
    "json",
    data_files=str(OUT_DIR / "probe_dataset_v1.jsonl"),
    split="train",
)
if len(loaded) != 100:
    raise RuntimeError("JSONL readback count mismatch")

for q in QUESTIONS:
    states = simulate(q["start"], q["delta"], q["depth"])
    if states != q["path"] or q["ans"] != states[-1]:
        raise RuntimeError(f"Reference answer mismatch: {q['question_id']}")

manifest = {
    "probe_version": PROBE_VERSION,
    "candidate": "B_numeric_array_pointer",
    "spec_version": "numeric-array-pointer-v1",
    "index_range": [0, N - 1],
    "min_abs_delta": MIN_ABS_DELTA,
    "depths": [1],
    "questions_per_depth": 100,
    "modes": ["DA", "CoT"],
    "planned_requests": 200,
    "data_seed": DATA_SEED,
    "model_seed": MODEL_SEED,
    "model_id": MODEL_ID,
    "system_prompts": SYSTEM_PROMPTS,
    "sampling_by_mode": SAMPLING_BY_MODE,
    "model_config": MODEL_KW,
    "scoring_version": SCORING_VERSION,
    "dataset_sha256": hashlib.sha256(dataset_text.encode("utf-8")).hexdigest(),
    "note": "p1_DA and p1_CoT are measured separately; no automatic promotion to A2.",
}
manifest["protocol_sha256"] = sha(manifest)
save_once(
    OUT_DIR / "probe_manifest_v1.json",
    json.dumps(manifest, ensure_ascii=False, indent=2) + "\n",
)

print("Frozen questions:", len(QUESTIONS))
print("Planned model requests:", manifest["planned_requests"])
print("Output directory:", OUT_DIR)
print("Existing attempt/result counts:", tuple(
    len(x) for x in request_status(OUT_DIR)
))

## 7. 加载 Qwen3-4B-Instruct-2507（Colab T4）

沿用上一轮已核实的 T4 配置，不自动降配、不切换模型。模型 revision 和实际运行环境会落盘。

In [ ]:
if importlib.util.find_spec("google.colab") is None:
    raise RuntimeError("Inference is restricted to the user's Google Colab T4 runtime.")

os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

check = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    check=True, capture_output=True, text=True,
)
print(check.stdout)
if "T4" not in check.stdout:
    raise RuntimeError("NVIDIA T4 not detected; do not switch hardware or model.")

import torch
import importlib.metadata
from huggingface_hub import HfApi

if not torch.cuda.is_available() or "T4" not in torch.cuda.get_device_name(0):
    raise RuntimeError("CUDA T4 is required")

attempts, existing = request_status(OUT_DIR)
if len(attempts) >= 200:
    raise RuntimeError("200 attempts already exist; skip model loading and inspect summary")

revision_path = OUT_DIR / "model_revision_v1.json"
if revision_path.exists():
    resolved = json.loads(revision_path.read_text())
    if resolved["model_id"] != MODEL_ID:
        raise RuntimeError("Existing model ID does not match")
else:
    resolved = {
        "model_id": MODEL_ID,
        "revision": HfApi().model_info(MODEL_ID).sha,
    }
    if not re.fullmatch(r"[0-9a-f]{40}", resolved["revision"] or ""):
        raise RuntimeError("Could not lock a model revision")
    save_once(
        revision_path,
        json.dumps(resolved, indent=2) + "\n",
    )

from vllm import LLM, SamplingParams

versions = {
    name: importlib.metadata.version(name)
    for name in ("vllm", "torch", "transformers", "datasets", "huggingface-hub")
}
identity = {
    "model_id": MODEL_ID,
    "model_revision": resolved["revision"],
    "versions": versions,
    "python": sys.version,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "gpu_memory_bytes": torch.cuda.get_device_properties(0).total_memory,
    "gpu_capability": list(torch.cuda.get_device_capability(0)),
    "nvidia_smi": check.stdout.strip(),
    "flashinfer_sampler": os.environ["VLLM_USE_FLASHINFER_SAMPLER"],
    "llm_config": MODEL_KW,
    "sampling_by_mode": SAMPLING_BY_MODE,
    "protocol_sha256": manifest["protocol_sha256"],
}
RUNTIME = {**identity, "environment_sha256": sha(identity)}
save_once(
    OUT_DIR / "runtime_v1.json",
    json.dumps(RUNTIME, ensure_ascii=False, indent=2) + "\n",
)
freeze = subprocess.run(
    [sys.executable, "-m", "pip", "freeze"],
    check=True, capture_output=True, text=True,
).stdout
save_once(OUT_DIR / "requirements_actual_v1.txt", freeze)

llm = LLM(
    model=MODEL_ID,
    revision=resolved["revision"],
    tokenizer_revision=resolved["revision"],
    **MODEL_KW,
)
tokenizer = llm.get_tokenizer()
save_once(
    OUT_DIR / "chat_template_v1.json",
    json.dumps(tokenizer.chat_template, ensure_ascii=False) + "\n",
)


def count_prompt(messages):
    return len(tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True
    ))


def generate_one(messages, mode, seed):
    params = SamplingParams(**SAMPLING_BY_MODE[mode], seed=seed)
    outputs = llm.chat(
        messages=messages,
        sampling_params=params,
        use_tqdm=False,
    )
    if len(outputs) != 1 or len(outputs[0].outputs) != 1:
        raise RuntimeError("Expected exactly one completion")
    item = outputs[0]
    completion = item.outputs[0]
    return {
        "raw_response": completion.text,
        "finish_reason": completion.finish_reason,
        "prompt_tokens": len(item.prompt_token_ids),
        "completion_tokens": len(completion.token_ids),
    }

print(json.dumps(RUNTIME, ensure_ascii=False, indent=2))

## 8. 首对请求检查（计入 200 次预算）

先发送第 1 道题的 DA 和 CoT。检查真实返回、评分字段与落盘状态。

In [ ]:
rows = run_requests(
    OUT_DIR, QUESTIONS, manifest, RUNTIME, generate_one, count_prompt, limit=2
)
if rows:
    print(json.dumps(rows[-1], ensure_ascii=False, indent=2))

## 9. 完成剩余请求

重跑此单元只发送未发送的请求。若存在已发送但结果未知的请求、截断或运行错误，会停止，不会自动重发。

In [ ]:
rows = run_requests(
    OUT_DIR, QUESTIONS, manifest, RUNTIME, generate_one, count_prompt
)
print("Saved result rows:", len(rows))

## 10. 汇总与 A1 判读

主准确率分母为全部已发送请求；解析失败计为未正确，同时单独报告解析率。DA 与 CoT 分开统计。

In [ ]:
attempts, results = request_status(OUT_DIR)
summary = summarize_probe(results, planned_per_mode=100)
summary["planned_requests"] = 200
summary["attempted_requests"] = len(attempts)
summary["not_attempted_requests"] = 200 - len(attempts)
summary["complete"] = len(attempts) == 200 and len(results) == 200

(OUT_DIR / "probe_summary_v1.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2) + "\n"
)
print(json.dumps(summary, ensure_ascii=False, indent=2))

## 11. 交付与边界

完成后带回以下文件：
- `probe_dataset_v1.jsonl`
- `probe_manifest_v1.json`
- `request_attempts_v1.jsonl`
- `raw_generations_v1.jsonl`
- `probe_results_v1.jsonl`
- `probe_summary_v1.json`
- `runtime_v1.json`
- `requirements_actual_v1.txt`

本 Notebook 不生成正式 d=1..20 数据集，也不运行约束集 Task B。只有人工审查确认 p1 窗口后，才进入 A3 的五个深度验证。